# Link gold queries with fused search + produce the dense/sparse/fused ablation table

Supersedes `link_gold_queries.ipynb`. That notebook used dense-only top-1 to link
the 280 `Gold_KB_final_280` questions to a `correct_answer_id`, and found a
52% category-mismatch rate (146/280) -- real signal, not noise, but dense-only
search is the weakest configuration available (no sparse leg, no rerank), so
some of that is exactly what Lever 3 exists to fix.

This notebook:
1. Runs **dense**, **sparse**, and **fused (RRF)** search for all 280 questions
2. Uses **fused's top-1** as the ground-truth `correct_answer_id` (better
   matching accuracy than dense-alone -- still flagged for human review, not
   silently trusted)
3. Measures Recall@1/5/20 for dense-only and sparse-only **against that fused
   ground truth** -- this is the Lever 3 ablation table, and it reuses the
   dense-only linking work as one leg of it rather than throwing it away

**Caveat, stated plainly:** fused's own "recall" against a ground truth defined
*by its own top-1* is close to tautological -- of course fused finds what fused
said was correct. The meaningful numbers here are how often dense-only and
sparse-only *independently* land on the same answer fused converged on. Once a
person has reviewed the flagged rows (category mismatch or low confidence),
rerun the Recall cells against the corrected `correct_answer_id` column for a
cleaner number.

Needs the same Qdrant credentials as before, and **Runtime -> Change runtime
type -> T4 GPU**.

In [ ]:
!pip install -q qdrant-client FlagEmbedding

In [ ]:
!rm -rf naari-ai
!git clone --branch sana/test-protection --depth 1 https://github.com/sana200420/naari-ai.git
%cd naari-ai

import sys
sys.path.insert(0, ".")

from retrieval.normalize import normalize_sd
from retrieval.search import HybridRetriever, reciprocal_rank_fusion

print("cloned + imported OK")

In [ ]:
import csv

with open("eval/gold_kb_final_280_raw.csv", encoding="utf-8-sig", newline="") as f:
    gold_rows = list(csv.DictReader(f))

with open("knowledge_base/Womens_Health_KB - 2000_final.csv", encoding="utf-8", newline="") as f:
    kb_rows = list(csv.DictReader(f))

sindhi_cats_order = []
seen_cats = set()
for r in kb_rows:
    if r["category"] not in seen_cats:
        sindhi_cats_order.append(r["category"]); seen_cats.add(r["category"])

EN_CATEGORY_ORDER = [
    "Menstrual Health & Periods", "Mental Health & Emotional Well-being",
    "Pregnancy & Maternal Health", "PCOS & Hormonal Health",
    "Women's Nutrition & Wellness", "Menopause & Menopausal Health",
    "Fertility & Reproductive Health", "Vaginal & Personal Hygiene",
]
sd_to_en_category = dict(zip(sindhi_cats_order, EN_CATEGORY_ORDER))

print(f"{len(gold_rows)} gold rows, {len(kb_rows)} KB rows, category map built")

In [ ]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
print("model loaded")

In [ ]:
from getpass import getpass
from qdrant_client import QdrantClient

QDRANT_URL = getpass("Qdrant cluster URL: ")
QDRANT_API_KEY = getpass("Qdrant API key: ")

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

def embed_fn(text):
    normalized = normalize_sd(text)
    out = model.encode([normalized], return_dense=True, return_sparse=True, return_colbert_vecs=False)
    return {"dense": out["dense_vecs"][0].tolist(), "sparse": out["lexical_weights"][0]}

retriever = HybridRetriever(client, embed_fn=embed_fn)
print("retriever ready, collection points:", client.get_collection("naari_ai_kb").points_count)

In [ ]:
TOP_K = 20
linked = []

for i, row in enumerate(gold_rows, start=1):
    query = row["Question"]

    dense_rows = retriever.dense_search(query, top_k=TOP_K)
    sparse_rows = retriever.sparse_search(query, top_k=TOP_K)

    dense_ids = [r["answer_id"] for r in dense_rows]
    sparse_ids = [r["answer_id"] for r in sparse_rows]
    fused = reciprocal_rank_fusion([dense_ids, sparse_ids])
    fused_ids = [aid for aid, _score in fused]

    row_by_id = {r["answer_id"]: r for r in dense_rows + sparse_rows}
    top1_id = fused_ids[0] if fused_ids else None
    top1_row = row_by_id.get(top1_id)
    top1_cat_en = sd_to_en_category.get(top1_row["category"], top1_row["category"]) if top1_row else ""

    linked.append({
        "query_id": f"gold_{row['ID']}",
        "query": query,
        "stated_category": row["Category"],
        "stated_subcategory": row["Subcategory"],
        "correct_answer_id": top1_id,
        "fused_top1_category_en": top1_cat_en,
        "category_match": (top1_cat_en == row["Category"]),
        "dense_top1_id": dense_ids[0] if dense_ids else None,
        "dense_top1_score": f"{dense_rows[0]['score']:.4f}" if dense_rows else "",
        "sparse_top1_id": sparse_ids[0] if sparse_ids else None,
        "sparse_top1_score": f"{sparse_rows[0]['score']:.4f}" if sparse_rows else "",
        "low_confidence": (dense_rows[0]["score"] < 0.6) if dense_rows else True,
        "gold_own_answer": row["Answer"],
        "gold_own_source": row["Source"],
        "_dense_ids": dense_ids,
        "_sparse_ids": sparse_ids,
        "_fused_ids": fused_ids,
    })

    if i % 40 == 0:
        print(f"linked {i}/{len(gold_rows)}")

print(f"done, {len(linked)} rows linked")

In [ ]:
import csv

OUT_PATH = "eval/gold_eval_280_linked.csv"
public_fields = [k for k in linked[0].keys() if not k.startswith("_")]
with open(OUT_PATH, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=public_fields)
    w.writeheader()
    for r in linked:
        w.writerow({k: r[k] for k in public_fields})

mismatches = [r for r in linked if not r["category_match"]]
low_conf = [r for r in linked if r["low_confidence"]]
print(f"wrote {OUT_PATH}")
print(f"category mismatches (fused top-1 vs stated category): {len(mismatches)}/{len(linked)}")
print(f"low-confidence dense matches (<0.6): {len(low_conf)}/{len(linked)}")
print("-> review these before treating correct_answer_id as final ground truth.")

## Ablation table: dense-only vs sparse-only vs fused, Recall@1/5/20

In [ ]:
def recall_at_k(ground_truth_key, ranked_key, k):
    hits = 0
    for r in linked:
        gt = r[ground_truth_key]
        ranked = r[ranked_key][:k]
        if gt in ranked:
            hits += 1
    return hits / len(linked)

results = {}
for leg, key in [("dense", "_dense_ids"), ("sparse", "_sparse_ids"), ("fused", "_fused_ids")]:
    results[leg] = {f"recall@{k}": recall_at_k("correct_answer_id", key, k) for k in (1, 5, 20)}

for leg, metrics in results.items():
    print(leg, metrics)

In [ ]:
import datetime

lines = []
lines.append("# Phase 1 dense/sparse/fused ablation -- gold_eval_280 (PROVISIONAL)\n")
lines.append(f"Generated: {datetime.datetime.utcnow().isoformat()}Z\n")
lines.append("\n")
lines.append("**Ground truth = fused search's own top-1 answer**, not independently "
             "human-verified yet. Fused's row is close to tautological (it is being "
             "measured against itself) -- the meaningful comparison is dense-only and "
             "sparse-only against that same reference point. "
             f"{len(mismatches)}/280 rows are flagged for category mismatch and "
             f"{len(low_conf)}/280 for low confidence; re-run after human review for a "
             "trustworthy final number.\n\n")
lines.append("| Leg | Recall@1 | Recall@5 | Recall@20 |\n")
lines.append("|---|---:|---:|---:|\n")
for leg in ("dense", "sparse", "fused"):
    m = results[leg]
    lines.append(f"| {leg} | {m['recall@1']:.3f} | {m['recall@5']:.3f} | {m['recall@20']:.3f} |\n")

with open("eval/results.md", "a", encoding="utf-8") as f:
    f.write("\n\n" + "".join(lines))

print("appended to eval/results.md")
print("".join(lines))

In [ ]:
from google.colab import files
files.download("eval/gold_eval_280_linked.csv")
files.download("eval/results.md")

## Next steps

1. Someone reviews every row flagged `category_match = False` or `low_confidence = True` in `eval/gold_eval_280_linked.csv`, correcting or dropping `correct_answer_id` as needed.
2. Re-run the Recall@K cell against the corrected column for a trustworthy final ablation table -- the one just written to `eval/results.md` is explicitly provisional.
3. Still confirm separately with whoever built `Gold_KB_final_280.csv` whether these 280 questions were drawn from real harvested speech -- linking to an answer_id doesn't resolve that.

# Final corrected ablation (post-review)

The 129 flagged rows have been human-reviewed and triaged: 105 confirmed OK as-is,
24 needed a closer look (manually cross-referenced against the full KB), and folded
back into `eval/gold_eval_280_linked.csv` as **9 corrected `correct_answer_id`
values + 5 dropped rows** (280 -> 275 rows). See `eval/gold_eval_280_needs_review_enriched.csv`
for the full reasoning per row.

Run this section fresh (needs a live model + Qdrant again, since the per-query
top-20 ranked lists from the first pass were never persisted to disk) to get the
final, non-provisional Recall@1/5/20 numbers against the corrected ground truth.

In [ ]:
import os

# Not relying on `git pull` any more -- it depends on this being a genuinely
# fresh runtime, and reconnecting Colab in a new tab does NOT give you a new
# kernel/VM if the old one is still alive server-side (same ipykernel PID as
# before is the tell). Unconditionally wipe and re-clone instead, exactly
# like the very first cell -- this is correct regardless of whatever state
# the VM's disk was already in.
%cd /content
!rm -rf naari-ai
!git clone --branch sana/test-protection --depth 1 https://github.com/sana200420/naari-ai.git
%cd naari-ai

import importlib
import retrieval.normalize
import retrieval.search
importlib.reload(retrieval.normalize)
importlib.reload(retrieval.search)
from retrieval.normalize import normalize_sd
from retrieval.search import HybridRetriever, reciprocal_rank_fusion

retriever = HybridRetriever(client, embed_fn=embed_fn)

import csv

with open("eval/gold_eval_280_linked.csv", encoding="utf-8-sig", newline="") as f:
    corrected_rows = list(csv.DictReader(f))

print(f"{len(corrected_rows)} corrected gold rows loaded (expect 275)")
assert len(corrected_rows) == 275, (
    f"got {len(corrected_rows)} rows, not 275. This cell re-clones unconditionally, "
    "so if you still see this, the fix genuinely isn't on GitHub yet -- check "
    "https://github.com/sana200420/naari-ai/blob/sana/test-protection/eval/gold_eval_280_linked.csv "
    "directly in a browser before re-running."
)

In [ ]:
TOP_K = 20
final_linked = []

for i, row in enumerate(corrected_rows, start=1):
    query = row["query"]
    gt_id = row["correct_answer_id"]

    dense_rows = retriever.dense_search(query, top_k=TOP_K)
    sparse_rows = retriever.sparse_search(query, top_k=TOP_K)

    dense_ids = [r["answer_id"] for r in dense_rows]
    sparse_ids = [r["answer_id"] for r in sparse_rows]
    fused_ids = [aid for aid, _score in reciprocal_rank_fusion([dense_ids, sparse_ids])]

    final_linked.append({
        "query_id": row["query_id"],
        "correct_answer_id": gt_id,
        "_dense_ids": dense_ids,
        "_sparse_ids": sparse_ids,
        "_fused_ids": fused_ids,
    })

    if i % 40 == 0:
        print(f"re-ranked {i}/{len(corrected_rows)}")

print(f"done, {len(final_linked)} rows re-ranked against corrected ground truth")

In [ ]:
def final_recall_at_k(ranked_key, k):
    hits = 0
    for r in final_linked:
        gt = str(r["correct_answer_id"])
        ranked = [str(x) for x in r[ranked_key][:k]]
        if gt in ranked:
            hits += 1
    return hits / len(final_linked)

final_results = {}
for leg, key in [("dense", "_dense_ids"), ("sparse", "_sparse_ids"), ("fused", "_fused_ids")]:
    final_results[leg] = {f"recall@{k}": final_recall_at_k(key, k) for k in (1, 5, 20)}

for leg, metrics in final_results.items():
    print(leg, metrics)

In [ ]:
import datetime

lines = []
lines.append("\n\n# Phase 1 dense/sparse/fused ablation -- gold_eval_275 (FINAL, human-reviewed)\n")
lines.append(f"Generated: {datetime.datetime.utcnow().isoformat()}Z\n\n")
lines.append("Ground truth = `eval/gold_eval_280_linked.csv` after human review of every "
              "flagged row (129/280): 105 confirmed correct as fused found them, 24 "
              "individually re-searched against the full KB producing 9 corrected "
              "`correct_answer_id` values and 5 rows dropped as having no usable KB match "
              "(280 -> 275 rows). Reasoning per reviewed row is in "
              "`eval/gold_eval_280_needs_review_enriched.csv`. This supersedes the "
              "provisional table above -- ground truth here is no longer fused's own "
              "top-1, so this is a real recall measurement, not a tautology.\n\n")
lines.append("| Leg | Recall@1 | Recall@5 | Recall@20 |\n")
lines.append("|---|---:|---:|---:|\n")
for leg in ("dense", "sparse", "fused"):
    m = final_results[leg]
    lines.append(f"| {leg} | {m['recall@1']:.3f} | {m['recall@5']:.3f} | {m['recall@20']:.3f} |\n")

with open("eval/results.md", "a", encoding="utf-8") as f:
    f.write("".join(lines))

print("appended FINAL table to eval/results.md")
print("".join(lines))

In [ ]:
from google.colab import files
files.download("eval/results.md")

# Lever 4: measure the English-rescue fraction

Checklist item asks: *of the queries where Sindhi-only retrieval fails, what
fraction does the English leg rescue?* Uses `_fused_ids` from the final
corrected pass above (Sindhi dense + Sindhi sparse, `lang="sd"` filter now
applied per the bug fix) to find the misses, then re-queries just those with
`HybridRetriever.cross_lingual_search()` (adds the NLLB-translated English
dense leg).

In [ ]:
!pip install -q transformers sentencepiece

# No git pull needed here -- the cell above already did a fresh re-clone.
from retrieval.translate import translate_sd_to_en

query_by_id = {row["query_id"]: row["query"] for row in corrected_rows}

sindhi_only_misses = [
    r for r in final_linked
    if str(r["correct_answer_id"]) not in [str(x) for x in r["_fused_ids"][:5]]
]
print(f"{len(sindhi_only_misses)}/{len(final_linked)} queries miss the correct "
      f"answer in the Sindhi-only fused top-5")

In [ ]:
rescued = 0
rescue_details = []

for i, r in enumerate(sindhi_only_misses, start=1):
    query = query_by_id[r["query_id"]]
    cl_rows = retriever.cross_lingual_search(query, top_k=5, translate_fn=translate_sd_to_en)
    cl_ids = [str(row["answer_id"]) for row in cl_rows]
    hit = str(r["correct_answer_id"]) in cl_ids
    rescued += hit
    rescue_details.append({"query_id": r["query_id"], "rescued": hit})

    if i % 10 == 0:
        print(f"{i}/{len(sindhi_only_misses)} processed, {rescued} rescued so far")

rescue_fraction = rescued / len(sindhi_only_misses) if sindhi_only_misses else float("nan")
print(f"\nLever 4 rescue fraction: {rescued}/{len(sindhi_only_misses)} = {rescue_fraction:.3f}")

In [ ]:
import datetime

lines = []
lines.append("\n\n# Lever 4 -- English-rescue fraction\n")
lines.append(f"Generated: {datetime.datetime.utcnow().isoformat()}Z\n\n")
lines.append(f"Of the {len(final_linked)} corrected gold queries, "
              f"**{len(sindhi_only_misses)} missed the correct answer in the Sindhi-only "
              f"fused top-5**. Adding the translated-query English leg "
              f"(`HybridRetriever.cross_lingual_search`) rescued "
              f"**{rescued}/{len(sindhi_only_misses)} ({rescue_fraction:.1%})** of those misses "
              "into its own top-5.\n\n")

with open("eval/results.md", "a", encoding="utf-8") as f:
    f.write("".join(lines))

print("appended to eval/results.md")
print("".join(lines))

# Verify bge-m3's prefix convention empirically

`docs/adr/0001-stack-decisions.md` states bge-m3 needs no `"query: "` /
`"passage: "` prefixes (unlike e5), but that was never actually tested --
Risk 1 in `docs/PLAYBOOKS.md` calls this out as *"the most common silent RAG
bug"* if gotten backwards.

**Method:** reuses the Sindhi dense top-20 candidate pool already fetched for
each query above (`_dense_ids`) rather than re-embedding the whole KB twice.
For each query where the correct answer is already in that pool, re-embeds
the query and the 20 candidates twice -- once unprefixed, once with
`"query: "`/`"passage: "` -- and compares the *rank* of the correct answer
within that same 20-candidate pool. This measures whether prefixes help or
hurt re-ranking; it is not a full-corpus retrieval test, so it's stated
alongside that caveat, not as a substitute for the real Recall@K numbers
above.

In [ ]:
import numpy as np

kb_by_id = {int(r["id"]): r for r in kb_rows}


def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


def rank_of_correct(query_text, candidate_ids, correct_id, query_prefix="", passage_prefix=""):
    q_norm = normalize_sd(query_prefix + query_text)
    q_vec = model.encode([q_norm], return_dense=True, return_sparse=False,
                          return_colbert_vecs=False)["dense_vecs"][0]

    cand_texts = [normalize_sd(passage_prefix + kb_by_id[int(cid)]["question"]) for cid in candidate_ids]
    cand_vecs = model.encode(cand_texts, return_dense=True, return_sparse=False,
                              return_colbert_vecs=False)["dense_vecs"]

    sims = [cosine(q_vec, v) for v in cand_vecs]
    ranked_ids = [str(cid) for cid, _ in sorted(zip(candidate_ids, sims), key=lambda p: -p[1])]
    return (ranked_ids.index(str(correct_id)) + 1) if str(correct_id) in ranked_ids else None


SAMPLE_N = 60
ranks_noprefix, ranks_prefixed = [], []

for i, r in enumerate(final_linked[:SAMPLE_N], start=1):
    query_text = query_by_id[r["query_id"]]
    candidate_ids = r["_dense_ids"][:20]
    if str(r["correct_answer_id"]) not in [str(c) for c in candidate_ids]:
        continue  # correct answer isn't in this pool -- can't compare ranks, skip

    ranks_noprefix.append(rank_of_correct(query_text, candidate_ids, r["correct_answer_id"]))
    ranks_prefixed.append(rank_of_correct(
        query_text, candidate_ids, r["correct_answer_id"],
        query_prefix="query: ", passage_prefix="passage: ",
    ))

    if i % 15 == 0:
        print(f"{i}/{SAMPLE_N}")

print(f"\ncompared {len(ranks_noprefix)} queries (of {SAMPLE_N} sampled, "
      f"the rest had no correct answer in their own top-20 pool)")


def recall_at(ranks, k):
    return sum(1 for r in ranks if r is not None and r <= k) / len(ranks)


print(f"no prefix  -- Recall@1: {recall_at(ranks_noprefix, 1):.3f}  Recall@5: {recall_at(ranks_noprefix, 5):.3f}")
print(f"prefixed   -- Recall@1: {recall_at(ranks_prefixed, 1):.3f}  Recall@5: {recall_at(ranks_prefixed, 5):.3f}")

In [ ]:
import datetime

verdict = (
    "confirms ADR 0001: no prefix does at least as well" if recall_at(ranks_noprefix, 5) >= recall_at(ranks_prefixed, 5)
    else "contradicts ADR 0001 -- prefixes ranked the correct answer higher, worth a closer look"
)

lines = []
lines.append("\n\n# bge-m3 prefix convention -- empirical check\n")
lines.append(f"Generated: {datetime.datetime.utcnow().isoformat()}Z\n\n")
lines.append(f"Re-ranked the existing Sindhi-dense top-20 candidate pool for "
              f"{len(ranks_noprefix)} gold queries, unprefixed vs with "
              "`\"query: \"`/`\"passage: \"` prefixes (see method note above -- this "
              "re-ranks an existing pool, it is not a full-corpus retrieval test).\n\n")
lines.append("| Convention | Recall@1 | Recall@5 |\n")
lines.append("|---|---:|---:|\n")
lines.append(f"| no prefix | {recall_at(ranks_noprefix, 1):.3f} | {recall_at(ranks_noprefix, 5):.3f} |\n")
lines.append(f"| query:/passage: prefix | {recall_at(ranks_prefixed, 1):.3f} | {recall_at(ranks_prefixed, 5):.3f} |\n\n")
lines.append(f"**Verdict:** {verdict}.\n\n")

with open("eval/results.md", "a", encoding="utf-8") as f:
    f.write("".join(lines))

print("appended to eval/results.md")
print("".join(lines))

files.download("eval/results.md")

# Item 8: cross-encoder reranking over the fused top-20

Checklist item asks for the Recall@1 improvement from reranking, measured and
committed. Uses `retrieval.rerank.rerank()` (`BAAI/bge-reranker-v2-m3`) over
`HybridRetriever.fused_search(query, top_k=20, lang="sd")` for each of the 275
corrected gold queries, per `docs/ROADMAP.md`'s pipeline (stage 05 reranks
`(query, Sindhi question)` pairs -- the Sindhi-only fused leg, not the
cross-lingual one).

Also records per-query latency (Risk 2's 3s p95 budget) and whether the
rerank score is better separated between hits and misses than raw RRF scores
are -- a first look at whether it's usable for Lever 5's confidence gate,
not the full threshold-tuning work itself (that's its own playbook item,
Phase 2).

In [ ]:
import time

from retrieval.rerank import rerank

rerank_before_ranks, rerank_after_ranks = [], []
before_top1_scores, after_top1_scores = [], []  # (score, was_correct) pairs
latencies = []

for i, row in enumerate(corrected_rows, start=1):
    query = row["query"]
    gt_id = str(row["correct_answer_id"])

    fused_rows = retriever.fused_search(query, top_k=20, leg_k=25, lang="sd")
    fused_ids = [str(r["answer_id"]) for r in fused_rows]
    before_rank = fused_ids.index(gt_id) + 1 if gt_id in fused_ids else None
    rerank_before_ranks.append(before_rank)
    if fused_rows:
        before_top1_scores.append((fused_rows[0]["score"], fused_ids[0] == gt_id))

    t0 = time.perf_counter()
    reranked_rows = rerank(query, fused_rows, top_k=20)
    latencies.append(time.perf_counter() - t0)

    reranked_ids = [str(r["answer_id"]) for r in reranked_rows]
    after_rank = reranked_ids.index(gt_id) + 1 if gt_id in reranked_ids else None
    rerank_after_ranks.append(after_rank)
    if reranked_rows:
        after_top1_scores.append((reranked_rows[0]["rerank_score"], reranked_ids[0] == gt_id))

    if i % 40 == 0:
        print(f"{i}/{len(corrected_rows)}")

print(f"done, {len(rerank_after_ranks)} queries reranked")

In [ ]:
import statistics


def recall_at(ranks, k):
    return sum(1 for r in ranks if r is not None and r <= k) / len(ranks)


before_r1, before_r5 = recall_at(rerank_before_ranks, 1), recall_at(rerank_before_ranks, 5)
after_r1, after_r5 = recall_at(rerank_after_ranks, 1), recall_at(rerank_after_ranks, 5)

print(f"fused (no rerank)  -- Recall@1: {before_r1:.3f}  Recall@5: {before_r5:.3f}")
print(f"reranked           -- Recall@1: {after_r1:.3f}  Recall@5: {after_r5:.3f}")

sorted_latencies = sorted(latencies)
p50 = sorted_latencies[len(sorted_latencies) // 2]
p95 = sorted_latencies[int(0.95 * len(sorted_latencies))]
print(f"\nrerank latency per query (20 candidates) -- mean: {statistics.mean(latencies)*1000:.1f}ms  "
      f"p50: {p50*1000:.1f}ms  p95: {p95*1000:.1f}ms")


def score_separation(scored_pairs):
    # RRF scores and rerank scores live on completely different scales, so
    # compare a standardised (Cohen's-d-style) gap, not the raw difference
    # of means -- that's the only way the two are comparable to each other.
    correct = [s for s, hit in scored_pairs if hit]
    incorrect = [s for s, hit in scored_pairs if not hit]
    if len(correct) < 2 or len(incorrect) < 2:
        return None
    pooled_std = statistics.pstdev(correct + incorrect)
    if pooled_std == 0:
        return None
    return (statistics.mean(correct) - statistics.mean(incorrect)) / pooled_std


before_sep = score_separation(before_top1_scores)
after_sep = score_separation(after_top1_scores)
print(f"\ntop-1 score separation, correct vs incorrect, in pooled std-devs "
      f"(bigger = more useful for a confidence gate; raw scores aren't comparable "
      f"across RRF and rerank since they live on different scales):")
print(f"  fused RRF:  {before_sep:.2f}" if before_sep is not None else "  fused: not enough of both classes to compare")
print(f"  reranked:   {after_sep:.2f}" if after_sep is not None else "  reranked: not enough of both classes to compare")

In [ ]:
import datetime

lines = []
lines.append("\n\n# Item 8 -- cross-encoder reranking over the fused top-20\n")
lines.append(f"Generated: {datetime.datetime.utcnow().isoformat()}Z\n\n")
lines.append("`retrieval.rerank.rerank()` (`BAAI/bge-reranker-v2-m3`) over "
              "`HybridRetriever.fused_search(top_k=20, lang=\"sd\")`, all 275 corrected "
              "gold queries.\n\n")
lines.append("| Stage | Recall@1 | Recall@5 |\n")
lines.append("|---|---:|---:|\n")
lines.append(f"| fused (no rerank) | {before_r1:.3f} | {before_r5:.3f} |\n")
lines.append(f"| reranked | {after_r1:.3f} | {after_r5:.3f} |\n\n")
lines.append(f"**Latency** (20 candidates/query): mean {statistics.mean(latencies)*1000:.1f}ms, "
              f"p50 {p50*1000:.1f}ms, p95 {p95*1000:.1f}ms -- against Risk 2's 3s p95 budget "
              "for the whole pipeline.\n\n")
lines.append("**Confidence-gate signal** (top-1 score separation between correct and "
              "incorrect answers, in pooled std-devs -- not directly comparable in raw "
              "units since RRF and rerank scores live on different scales):\n")
lines.append(f"- fused RRF: {before_sep:.2f}\n" if before_sep is not None else "- fused RRF: not enough of both classes to compare\n")
lines.append(f"- reranked: {after_sep:.2f}\n\n" if after_sep is not None else "- reranked: not enough of both classes to compare\n\n")

with open("eval/results.md", "a", encoding="utf-8") as f:
    f.write("".join(lines))

print("appended to eval/results.md")
print("".join(lines))

files.download("eval/results.md")

# Diagnostic: is the reranking drop real or a bug?

Reranking took fused Recall@1 from 0.967 down to 0.385 -- **worse than even
the weakest single leg (sparse-only, 0.542)**. If the reranker had any real
signal at all, it shouldn't score below a leg it never even sees. That's
more consistent with an ordering/pairing bug than with "the reranker
genuinely understands Sindhi worse than embedding similarity does."

Two checks:
1. **Sanity check the model+function in isolation** -- score an identical
   (query, query) pair (should score high) against an obviously unrelated
   pair (should score low), independent of anything from the loop above.
2. **Look at actual regressions** -- for a handful of queries where fused
   had the correct answer at rank 1 but reranking pushed it down, print the
   correct candidate's question + score next to whatever the reranker
   preferred instead, so we can read whether it's a real (if surprising)
   semantic disagreement or nonsense.

In [ ]:
regressions = [
    (row, before, after)
    for row, before, after in zip(corrected_rows, rerank_before_ranks, rerank_after_ranks)
    if before == 1 and after != 1
]
print(f"{len(regressions)} queries regressed from rank 1 (fused) to somewhere else (reranked)\n")

for row, before, after in regressions[:8]:
    query = row["query"]
    gt_id = str(row["correct_answer_id"])

    fused_rows = retriever.fused_search(query, top_k=20, leg_k=25, lang="sd")
    reranked_rows = rerank(query, fused_rows, top_k=20)

    correct_fused = next((r for r in fused_rows if str(r["answer_id"]) == gt_id), None)
    correct_reranked = next((r for r in reranked_rows if str(r["answer_id"]) == gt_id), None)
    top1_reranked = reranked_rows[0]

    print(f"query: {query}")
    if correct_fused:
        print(f"  correct answer (id={gt_id}): {correct_fused['question']}")
        print(f"    fused score: {correct_fused['score']:.4f} (was rank 1)")
    if correct_reranked:
        print(f"    rerank score: {correct_reranked['rerank_score']:.4f}  -> new rank: {after}")
    else:
        print(f"    correct answer fell OUT of the reranked top-20 entirely")
    print(f"  reranker preferred instead (id={top1_reranked['answer_id']}): {top1_reranked['question']}")
    print(f"    rerank score: {top1_reranked['rerank_score']:.4f}")
    print()

In [ ]:
from retrieval.rerank import _get_model

reranker_model = _get_model()

sanity_pairs = [
    ["ماهواري ڪيتري دير هلندي آهي؟", "ماهواري ڪيتري دير هلندي آهي؟"],  # identical -- expect high
    ["ماهواري ڪيتري دير هلندي آهي؟", "پاڪستان جو گاديءَ جو شهر ڪهڙو آهي؟"],  # unrelated -- expect low
]
sanity_scores = reranker_model.compute_score(sanity_pairs, normalize=True)
print("identical-pair score (expect high, close to 1):", sanity_scores[0])
print("unrelated-pair score (expect low, close to 0):  ", sanity_scores[1])
assert sanity_scores[0] > sanity_scores[1], (
    "model+function disagree with themselves on a trivial case -- "
    "something is broken before we even get to real data"
)
print("\nsanity check passed: compute_score behaves as expected in isolation")